<a href="https://colab.research.google.com/github/nishattasnim15215/AI-Agent-LLM-with-SQL-Groq/blob/main/AI%20Agent%20LLM%20with%20SQL%20Groq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import Required Libraries & Get API Key
import os            # For accessing environment variables like API keys
import sqlite3       # For creating and interacting with an SQLite database
import requests      # To make HTTP requests to the Groq API
import json          # To format and parse request/response data

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except (ImportError, Exception):
    GROQ_API_KEY = os.environ.get('GROQ_API_KEY')

assert GROQ_API_KEY, "GROQ_API_KEY not found."

In [ ]:
# This function creates a local SQLite database named 'school.db' and a table called 'students'
def setup_database():
    conn = sqlite3.connect('school.db')       # Connect to or create the SQLite database file
    cursor = conn.cursor()                    # Create a cursor object to execute SQL commands

    # Create the students table if it doesn't already exist
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS students (
            roll_number INTEGER PRIMARY KEY,
            name TEXT,
            department TEXT,
            grade REAL,
            attendance_percent REAL,
            graduation_year INTEGER
        )
    ''')

    # Add 5 sample student records
    sample_data = [
        (1, 'Alice Johnson', 'Science', 88.5, 92.0, 2026),
        (2, 'Bob Lee', 'Arts', 76.0, 85.5, 2025),
        (3, 'Carlos Patel', 'Engineering', 91.2, 98.0, 2026),
        (4, 'Diana Rose', 'Science', 67.8, 73.2, 2024),
        (5, 'Eva Green', 'Engineering', 82.0, 89.9, 2025)
    ]

    # Insert data into the students table (IGNORE skips if already present)
    cursor.executemany("INSERT OR IGNORE INTO students VALUES (?, ?, ?, ?, ?, ?)", sample_data)

    conn.commit()     # Save the changes to the database
    conn.close()      # Close the connection

# Run the function once to initialize the database
setup_database()
print("Student database created.")

Student database created.


In [ ]:
# Define the Database Schema for the LLM
# This function returns a description of the 'students' table so the LLM understands it
def get_schema():
    return """
    Table: students
    Columns:
    - roll_number (INTEGER PRIMARY KEY)
    - name (TEXT)
    - department (TEXT)
    - grade (REAL)
    - attendance_percent (REAL)
    - graduation_year (INTEGER)
    """

In [ ]:
# This function sends a prompt to Groq's LLM and gets back an SQL query
def call_groq_llm(prompt):
    url = "https://api.groq.com/openai/v1/chat/completions"

    # Set the HTTP headers including authorization
    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",        # API key for access
        "Content-Type": "application/json"                # Body will be JSON format
    }

    # Create the full prompt using schema + question
    payload = {
        "model": "llama-3.3-70b-versatile",
        "messages": [
            {
                "role": "system",
                "content": (
                    f"You are a helpful SQL assistant. Use this schema:\n{get_schema()}\n"
                    "Return ONLY the raw SQL query. No explanations, no markdown code fences, no comments."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": 0                                  # Deterministic output
    }

    # Send request to Groq API
    response = requests.post(url, headers=headers, data=json.dumps(payload))

    # If request was successful, extract and clean the SQL query from the response
    if response.status_code == 200:
        result = response.json()
        sql = result['choices'][0]['message']['content'].strip()

        # Strip markdown code fences if the model added them anyway
        if sql.startswith("```"):
            lines = sql.split("\n")
            # Drop the first line (```sql or ```) and the last line (```)
            lines = [l for l in lines if not l.strip().startswith("```")]
            sql = "\n".join(lines).strip()
        return sql
    else:
        raise Exception(f"Groq API error: {response.status_code} - {response.text}")

In [ ]:
# Block potentially dangerous SQL commands like DELETE, DROP, etc.
def validate_sql(sql):
    sql_lower = sql.lower()  # Convert SQL to lowercase for easier checking
    if any(cmd in sql_lower for cmd in ['drop', 'delete', 'update', 'insert', 'alter', 'truncate']):
        raise ValueError("Unsafe query blocked. Only SELECT statements are allowed.")
    return sql

In [ ]:
# This function executes a safe SQL query and returns the results
def execute_query(sql):
    sql = validate_sql(sql)              # Check for safety
    conn = sqlite3.connect('school.db')  # Connect to the database
    try:
        cursor = conn.cursor()
        cursor.execute(sql)              # Run the SQL command
        return cursor.fetchall()         # Return all rows of the result
    except Exception as e:
        return f"Error: {str(e)}"        # Handle and return any error message
    finally:
        conn.close()                     # Always close the connection

In [ ]:
# This function formats the database results into a clean table-like string
def format_results(results):
    if isinstance(results, str):
        return results                   # If it's an error message, return as is
    if not results:
        return "No results found."       # Handle empty query results

    lines = []
    for row in results:
        # Convert each row into a string, formatting floats to 2 decimal places
        lines.append("\t".join(str(item) if not isinstance(item, float) else f"{item:.2f}" for item in row))
    return "\n".join(lines)              # Join all rows with newlines

In [ ]:
# This is the final agent function that connects everything together (Full Agent: Ask, Convert, Execute, Format)
def query_agent(question):
    try:
        # Convert natural language to SQL using Groq's LLM
        sql = call_groq_llm(f"Generate SQL for: {question}")
        print(f"Generated SQL:\n{sql}\n")

        # Run the SQL query and return formatted results
        results = execute_query(sql)
        return format_results(results)
    except Exception as e:
        return f"Error: {str(e)}"

In [ ]:
# Test the Agent with Example Questions
test_questions = [
    "List all students",
    "Show students with grade above 85",
    "Who is graduating in 2025?",
    "What is the average attendance for each department?",
    "DROP TABLE students"  # This should be blocked by validate_sql()
]

# Loop through questions and show results
for question in test_questions:
    print(f"\nQuestion: {question}")
    print(f"Answer:\n{query_agent(question)}")


Question: List all students
Generated SQL:
SELECT * FROM students

Answer:
1	Alice Johnson	Science	88.50	92.00	2026
2	Bob Lee	Arts	76.00	85.50	2025
3	Carlos Patel	Engineering	91.20	98.00	2026
4	Diana Rose	Science	67.80	73.20	2024
5	Eva Green	Engineering	82.00	89.90	2025

Question: Show students with grade above 85
Generated SQL:
SELECT * FROM students WHERE grade > 85

Answer:
1	Alice Johnson	Science	88.50	92.00	2026
3	Carlos Patel	Engineering	91.20	98.00	2026

Question: Who is graduating in 2025?
Generated SQL:
SELECT name FROM students WHERE graduation_year = 2025

Answer:
Bob Lee
Eva Green

Question: What is the average attendance for each department?
Generated SQL:
SELECT department, AVG(attendance_percent) FROM students GROUP BY department

Answer:
Arts	85.50
Engineering	93.95
Science	82.60

Question: DROP TABLE students
Generated SQL:
DROP TABLE students

Answer:
Error: Unsafe query blocked. Only SELECT statements are allowed.


In [ ]:
question = input("Enter your question for the SQL agent: ")
print(f"\nQuestion: {question}")
print(f"Answer:\n{query_agent(question)}")

Enter your question for the SQL agent: What are the names of students in the Science department?

Question: What are the names of students in the Science department?
Generated SQL:
SELECT name FROM students WHERE department = 'Science'

Answer:
Alice Johnson
Diana Rose
